In [1]:

import numpy as np
import pandas as pd
import glob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, log_loss
)

# Load SMS Spam Dataset
files = glob.glob("/kaggle/input/**/spam.csv", recursive=True)

if not files:
    raise FileNotFoundError("Please add SMS Spam Collection Dataset")

df = pd.read_csv(files[0], encoding="latin-1")

# Keep only message and label
df = df[["v1", "v2"]].copy()
df.columns = ["label", "message"]

# Spam = 1, Not Spam = 0
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Split data into training and testing
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# Convert messages into numeric features
vectorizer = TfidfVectorizer(max_features=1000)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print("Total Messages:", len(df))
print("Spam:", (df["label"] == 1).sum())
print("Not Spam:", (df["label"] == 0).sum())
print("Training:", X_train.shape)
print("Testing:", X_test.shape)


Total Messages: 5572
Spam: 747
Not Spam: 4825
Training: (4457, 1000)
Testing: (1115, 1000)


In [2]:

# ==========================================
# Optimizer 1: Batch Gradient Descent
# Logistic Regression - Spam / Not Spam
# ==========================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

# Sigmoid function
def sigmoid_gd(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Initialize weights and bias
weights_gd = np.zeros(X_train.shape[1])
bias_gd = 0.0

learning_rate_gd = 0.5
epochs_gd = 300

# Train Logistic Regression with Batch GD
for epoch in range(epochs_gd):

    # Calculate probabilities
    probabilities = sigmoid_gd(X_train @ weights_gd + bias_gd)

    # Calculate gradients
    error = probabilities - y_train

    dw = np.asarray(X_train.T @ error).ravel() / len(y_train)
    db = np.mean(error)

    # Update weights and bias
    weights_gd -= learning_rate_gd * dw
    bias_gd -= learning_rate_gd * db

# Predict test messages
gd_probabilities = sigmoid_gd(X_test @ weights_gd + bias_gd)

# Convert probabilities into classes
gd_predictions = (gd_probabilities >= 0.5).astype(int)

# Evaluate the model
print("Gradient Descent Results")
print("Accuracy:", round(accuracy_score(y_test, gd_predictions), 4))
print("Precision:", round(precision_score(y_test, gd_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, gd_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, gd_predictions, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, gd_probabilities), 4))

# Display Spam and Not Spam predictions
print("\nPredicted Spam:", np.sum(gd_predictions == 1))
print("Predicted Not Spam:", np.sum(gd_predictions == 0))

# Display sample classification results
gd_output = pd.DataFrame({
    "Message": X_test_text.to_numpy(),
    "Actual": np.where(y_test == 1, "Spam", "Not Spam"),
    "Predicted": np.where(gd_predictions == 1, "Spam", "Not Spam")
})

display(gd_output.head(10))


Gradient Descent Results
Accuracy: 0.8664
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Loss: 0.3079

Predicted Spam: 0
Predicted Not Spam: 1115


,Message,Actual,Predicted
0,"Oh right, ok. I'll make sure that i do loads o...",Not Spam,Not Spam
1,I am in tirupur. call you da.,Not Spam,Not Spam
2,No that just means you have a fat head,Not Spam,Not Spam
3,"You have won ?1,000 cash or a ?2,000 prize! To...",Spam,Not Spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning ...,Not Spam,Not Spam
5,Friendship poem: Dear O Dear U R Not Near But ...,Not Spam,Not Spam
6,Wot about on wed nite I am 3 then but only til 9!,Not Spam,Not Spam
7,Dont talk to him ever ok its my word.,Not Spam,Not Spam
8,Congrats kano..whr s the treat maga?,Not Spam,Not Spam
9,Eh u remember how 2 spell his name... Yes i di...,Not Spam,Not Spam


In [3]:

# ==========================================
# Optimizer 2: Stochastic Gradient Descent
# Logistic Regression - Spam / Not Spam
# ==========================================

# Initialize parameters
weights_sgd = np.zeros(X_train.shape[1])
bias_sgd = 0.0

learning_rate_sgd = 0.01
epochs_sgd = 20

rng_sgd = np.random.default_rng(42)

# Train Logistic Regression using SGD
for epoch in range(epochs_sgd):
    indices = rng_sgd.permutation(X_train.shape[0])

    for i in indices:
        X_i = X_train[i:i+1]
        y_i = y_train[i:i+1]

        z = X_i @ weights_sgd + bias_sgd
        probabilities = 1 / (
            1 + np.exp(-np.clip(z, -500, 500))
        )

        error = probabilities - y_i

        dw = np.asarray(X_i.T @ error).ravel()
        db = np.mean(error)

        # Update parameters
        weights_sgd -= learning_rate_sgd * dw
        bias_sgd -= learning_rate_sgd * db

# Predict test messages
z_test = X_test @ weights_sgd + bias_sgd
sgd_probabilities = 1 / (
    1 + np.exp(-np.clip(z_test, -500, 500))
)
sgd_predictions = (sgd_probabilities >= 0.5).astype(int)

# Evaluation metrics
print("SGD Results")
print("Accuracy:", round(accuracy_score(y_test, sgd_predictions), 4))
print("Precision:", round(precision_score(y_test, sgd_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, sgd_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, sgd_predictions, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, sgd_probabilities), 4))

# Spam and Not Spam predictions
print("\nPredicted Spam:", np.sum(sgd_predictions == 1))
print("Predicted Not Spam:", np.sum(sgd_predictions == 0))

# Display sample predictions
sgd_output = pd.DataFrame({
    "Message": X_test_text.to_numpy(),
    "Actual": np.where(y_test == 1, "Spam", "Not Spam"),
    "Predicted": np.where(sgd_predictions == 1, "Spam", "Not Spam")
})

display(sgd_output.head(10))


SGD Results
Accuracy: 0.9534
Precision: 0.9899
Recall: 0.6577
F1 Score: 0.7903
Loss: 0.1514

Predicted Spam: 99
Predicted Not Spam: 1016


,Message,Actual,Predicted
0,"Oh right, ok. I'll make sure that i do loads o...",Not Spam,Not Spam
1,I am in tirupur. call you da.,Not Spam,Not Spam
2,No that just means you have a fat head,Not Spam,Not Spam
3,"You have won ?1,000 cash or a ?2,000 prize! To...",Spam,Spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning ...,Not Spam,Not Spam
5,Friendship poem: Dear O Dear U R Not Near But ...,Not Spam,Not Spam
6,Wot about on wed nite I am 3 then but only til 9!,Not Spam,Not Spam
7,Dont talk to him ever ok its my word.,Not Spam,Not Spam
8,Congrats kano..whr s the treat maga?,Not Spam,Not Spam
9,Eh u remember how 2 spell his name... Yes i di...,Not Spam,Not Spam


In [4]:

# ==========================================
# Optimizer 3: Mini-Batch Gradient Descent
# Logistic Regression - Spam / Not Spam
# ==========================================

# Initialize parameters
weights_mb = np.zeros(X_train.shape[1])
bias_mb = 0.0

learning_rate_mb = 0.01
epochs_mb = 50
batch_size_mb = 32

rng_mb = np.random.default_rng(42)

# Train using Mini-Batch Gradient Descent
for epoch in range(epochs_mb):
    indices = rng_mb.permutation(X_train.shape[0])

    for start in range(0, X_train.shape[0], batch_size_mb):
        batch_indices = indices[start:start + batch_size_mb]

        X_batch = X_train[batch_indices]
        y_batch = y_train[batch_indices]

        z = X_batch @ weights_mb + bias_mb
        probabilities = 1 / (1 + np.exp(-np.clip(z, -500, 500)))

        error = probabilities - y_batch

        dw = np.asarray(X_batch.T @ error).ravel() / len(y_batch)
        db = np.mean(error)

        # Update weights and bias
        weights_mb -= learning_rate_mb * dw
        bias_mb -= learning_rate_mb * db

# Predict on test data
z_test_mb = X_test @ weights_mb + bias_mb
mb_probabilities = 1 / (
    1 + np.exp(-np.clip(z_test_mb, -500, 500))
)
mb_predictions = (mb_probabilities >= 0.5).astype(int)

# Evaluation metrics
print("Mini-Batch GD Results")
print("Accuracy:", round(accuracy_score(y_test, mb_predictions), 4))
print("Precision:", round(precision_score(y_test, mb_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, mb_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, mb_predictions, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, mb_probabilities), 4))

# Spam and Not Spam predictions
print("\nPredicted Spam:", np.sum(mb_predictions == 1))
print("Predicted Not Spam:", np.sum(mb_predictions == 0))

# Display classification examples
mb_output = pd.DataFrame({
    "Message": X_test_text.to_numpy(),
    "Actual": np.where(y_test == 1, "Spam", "Not Spam"),
    "Predicted": np.where(mb_predictions == 1, "Spam", "Not Spam")
})

display(mb_output.head(10))


Mini-Batch GD Results
Accuracy: 0.8664
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Loss: 0.3499

Predicted Spam: 0
Predicted Not Spam: 1115


,Message,Actual,Predicted
0,"Oh right, ok. I'll make sure that i do loads o...",Not Spam,Not Spam
1,I am in tirupur. call you da.,Not Spam,Not Spam
2,No that just means you have a fat head,Not Spam,Not Spam
3,"You have won ?1,000 cash or a ?2,000 prize! To...",Spam,Not Spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning ...,Not Spam,Not Spam
5,Friendship poem: Dear O Dear U R Not Near But ...,Not Spam,Not Spam
6,Wot about on wed nite I am 3 then but only til 9!,Not Spam,Not Spam
7,Dont talk to him ever ok its my word.,Not Spam,Not Spam
8,Congrats kano..whr s the treat maga?,Not Spam,Not Spam
9,Eh u remember how 2 spell his name... Yes i di...,Not Spam,Not Spam


In [5]:

# ==========================================
# Optimizer 4: RMSprop
# Logistic Regression - Spam / Not Spam
# ==========================================

# Initialize parameters
weights_rms = np.zeros(X_train.shape[1])
bias_rms = 0.0

learning_rate_rms = 0.001
epochs_rms = 50
batch_size_rms = 32
beta_rms = 0.9
epsilon_rms = 1e-8

# Initialize squared gradient averages
v_w_rms = np.zeros(X_train.shape[1])
v_b_rms = 0.0

rng_rms = np.random.default_rng(42)

# Train Logistic Regression using RMSprop
for epoch in range(epochs_rms):
    indices = rng_rms.permutation(X_train.shape[0])

    for start in range(0, X_train.shape[0], batch_size_rms):
        batch_indices = indices[start:start + batch_size_rms]

        X_batch = X_train[batch_indices]
        y_batch = y_train[batch_indices]

        # Predictions
        z = X_batch @ weights_rms + bias_rms
        probabilities = 1 / (1 + np.exp(-np.clip(z, -500, 500)))

        # Gradients
        error = probabilities - y_batch
        dw = np.asarray(X_batch.T @ error).ravel() / len(y_batch)
        db = np.mean(error)

        # RMSprop moving averages
        v_w_rms = beta_rms * v_w_rms + (1 - beta_rms) * dw**2
        v_b_rms = beta_rms * v_b_rms + (1 - beta_rms) * db**2

        # Update parameters
        weights_rms -= learning_rate_rms * dw / (
            np.sqrt(v_w_rms) + epsilon_rms
        )
        bias_rms -= learning_rate_rms * db / (
            np.sqrt(v_b_rms) + epsilon_rms
        )

# Test predictions
z_test_rms = X_test @ weights_rms + bias_rms
rms_probabilities = 1 / (
    1 + np.exp(-np.clip(z_test_rms, -500, 500))
)
rms_predictions = (rms_probabilities >= 0.5).astype(int)

# Evaluation metrics
print("RMSprop Results")
print("Accuracy:", round(accuracy_score(y_test, rms_predictions), 4))
print("Precision:", round(precision_score(y_test, rms_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, rms_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, rms_predictions, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, rms_probabilities), 4))

# Spam and Not Spam predictions
print("\nPredicted Spam:", np.sum(rms_predictions == 1))
print("Predicted Not Spam:", np.sum(rms_predictions == 0))

# Display classification examples
rms_output = pd.DataFrame({
    "Message": X_test_text.to_numpy(),
    "Actual": np.where(y_test == 1, "Spam", "Not Spam"),
    "Predicted": np.where(rms_predictions == 1, "Spam", "Not Spam")
})

display(rms_output.head(10))


RMSprop Results
Accuracy: 0.9776
Precision: 0.9769
Recall: 0.8523
F1 Score: 0.9104
Loss: 0.083

Predicted Spam: 130
Predicted Not Spam: 985


,Message,Actual,Predicted
0,"Oh right, ok. I'll make sure that i do loads o...",Not Spam,Not Spam
1,I am in tirupur. call you da.,Not Spam,Not Spam
2,No that just means you have a fat head,Not Spam,Not Spam
3,"You have won ?1,000 cash or a ?2,000 prize! To...",Spam,Spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning ...,Not Spam,Not Spam
5,Friendship poem: Dear O Dear U R Not Near But ...,Not Spam,Not Spam
6,Wot about on wed nite I am 3 then but only til 9!,Not Spam,Not Spam
7,Dont talk to him ever ok its my word.,Not Spam,Not Spam
8,Congrats kano..whr s the treat maga?,Not Spam,Not Spam
9,Eh u remember how 2 spell his name... Yes i di...,Not Spam,Not Spam


In [6]:

# ==========================================
# Optimizer 5: Adam
# Logistic Regression - Spam / Not Spam
# ==========================================

# Initialize parameters
weights_adam = np.zeros(X_train.shape[1])
bias_adam = 0.0

learning_rate_adam = 0.001
epochs_adam = 50
batch_size_adam = 32

beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

# Initialize first and second moments
m_w = np.zeros(X_train.shape[1])
v_w = np.zeros(X_train.shape[1])
m_b = 0.0
v_b = 0.0

t = 0
rng_adam = np.random.default_rng(42)

# Train Logistic Regression using Adam
for epoch in range(epochs_adam):
    indices = rng_adam.permutation(X_train.shape[0])

    for start in range(0, X_train.shape[0], batch_size_adam):
        batch_indices = indices[start:start + batch_size_adam]

        X_batch = X_train[batch_indices]
        y_batch = y_train[batch_indices]

        # Calculate probabilities
        z = X_batch @ weights_adam + bias_adam
        probabilities = 1 / (1 + np.exp(-np.clip(z, -500, 500)))

        # Calculate gradients
        error = probabilities - y_batch
        dw = np.asarray(X_batch.T @ error).ravel() / len(y_batch)
        db = np.mean(error)

        t += 1

        # First moment estimates
        m_w = beta1 * m_w + (1 - beta1) * dw
        m_b = beta1 * m_b + (1 - beta1) * db

        # Second moment estimates
        v_w = beta2 * v_w + (1 - beta2) * dw**2
        v_b = beta2 * v_b + (1 - beta2) * db**2

        # Bias correction
        m_w_hat = m_w / (1 - beta1**t)
        m_b_hat = m_b / (1 - beta1**t)
        v_w_hat = v_w / (1 - beta2**t)
        v_b_hat = v_b / (1 - beta2**t)

        # Update parameters
        weights_adam -= learning_rate_adam * m_w_hat / (
            np.sqrt(v_w_hat) + epsilon
        )
        bias_adam -= learning_rate_adam * m_b_hat / (
            np.sqrt(v_b_hat) + epsilon
        )

# Predict test messages
z_test_adam = X_test @ weights_adam + bias_adam
adam_probabilities = 1 / (
    1 + np.exp(-np.clip(z_test_adam, -500, 500))
)
adam_predictions = (adam_probabilities >= 0.5).astype(int)

# Evaluation metrics
print("Adam Results")
print("Accuracy:", round(accuracy_score(y_test, adam_predictions), 4))
print("Precision:", round(precision_score(y_test, adam_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, adam_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, adam_predictions, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, adam_probabilities), 4))

# Spam and Not Spam predictions
print("\nPredicted Spam:", np.sum(adam_predictions == 1))
print("Predicted Not Spam:", np.sum(adam_predictions == 0))

# Display classification examples
adam_output = pd.DataFrame({
    "Message": X_test_text.to_numpy(),
    "Actual": np.where(y_test == 1, "Spam", "Not Spam"),
    "Predicted": np.where(adam_predictions == 1, "Spam", "Not Spam")
})

display(adam_output.head(10))


Adam Results
Accuracy: 0.9803
Precision: 0.9922
Recall: 0.8591
F1 Score: 0.9209
Loss: 0.0839

Predicted Spam: 129
Predicted Not Spam: 986


,Message,Actual,Predicted
0,"Oh right, ok. I'll make sure that i do loads o...",Not Spam,Not Spam
1,I am in tirupur. call you da.,Not Spam,Not Spam
2,No that just means you have a fat head,Not Spam,Not Spam
3,"You have won ?1,000 cash or a ?2,000 prize! To...",Spam,Spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning ...,Not Spam,Not Spam
5,Friendship poem: Dear O Dear U R Not Near But ...,Not Spam,Not Spam
6,Wot about on wed nite I am 3 then but only til 9!,Not Spam,Not Spam
7,Dont talk to him ever ok its my word.,Not Spam,Not Spam
8,Congrats kano..whr s the treat maga?,Not Spam,Not Spam
9,Eh u remember how 2 spell his name... Yes i di...,Not Spam,Not Spam
